<a href="https://colab.research.google.com/github/databyhuseyn/DeepLearning/blob/main/Emotions_AI102.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!curl -L -o emotion-recognition-dataset.zip https://www.kaggle.com/api/v1/datasets/download/sujaykapadnis/emotion-recognition-dataset

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 2027M  100 2027M    0     0   202M      0  0:00:10  0:00:10 --:--:--  231M


In [2]:
!ls

emotion-recognition-dataset.zip  sample_data


In [3]:
from zipfile import ZipFile

In [4]:
with ZipFile('emotion-recognition-dataset.zip', 'r') as f:
  f.extractall()

In [5]:
!rm -rf dataset/Ahegao

In [6]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [7]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [8]:
train_dataset = datasets.ImageFolder(
    root='/content/dataset',
    transform=transform
)

In [9]:
class_names = train_dataset.classes
class_names

['Angry', 'Happy', 'Neutral', 'Sad', 'Surprise']

In [10]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [11]:
import torchvision.models as models
import torch

In [12]:
weights = models.ResNet34_Weights.IMAGENET1K_V1
resnet34 = models.resnet34(weights=weights)

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 144MB/s]


In [13]:
if torch.cuda.is_available():
  device='cuda'
elif torch.backends.mps.is_available():
  device='mps'
else:
  device='cpu'

In [15]:
[key for key, value in resnet34.named_children()]

['conv1',
 'bn1',
 'relu',
 'maxpool',
 'layer1',
 'layer2',
 'layer3',
 'layer4',
 'avgpool',
 'fc']

In [16]:
import torch.nn as nn

In [17]:
resnet34.fc = nn.Linear(in_features=512, out_features=len(class_names), bias=True)

In [18]:
for param in resnet34.parameters():
  param.requires_grad = False

for param in resnet34.fc.parameters():
  param.requires_grad = True

In [19]:
def train(model, optimizer, criterion, train_loader, n_epochs:int = 10):
  for epoch in range(n_epochs):
    total_loss = 0.
    for image, label in train_loader:
      image, label = image.to(device), label.to(device)
      pred = model(image)
      loss = criterion(pred, label)
      total_loss += loss.item()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

    print(f'Epoch: {epoch+1}/{n_epochs}, Loss: {total_loss / len(train_loader)}')


In [20]:
optimizer = torch.optim.Adam(resnet34.parameters(), lr=1e-4)
xentropy = nn.CrossEntropyLoss()
N_EPOCHS = 20

In [22]:
resnet34.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [23]:
train(resnet34, optimizer, xentropy, train_loader, N_EPOCHS)

Epoch: 1/20, Loss: 1.4569887652525453
Epoch: 2/20, Loss: 1.287650764790351
Epoch: 3/20, Loss: 1.2057838238140928
Epoch: 4/20, Loss: 1.1624233760106724
Epoch: 5/20, Loss: 1.1284323838526893
Epoch: 6/20, Loss: 1.1070207446947227
Epoch: 7/20, Loss: 1.087957212876846
Epoch: 8/20, Loss: 1.073244861155882
Epoch: 9/20, Loss: 1.0620825468424724
Epoch: 10/20, Loss: 1.0527135994402284
Epoch: 11/20, Loss: 1.0427594572439323
Epoch: 12/20, Loss: 1.0375193556595277
Epoch: 13/20, Loss: 1.0290791380298512
Epoch: 14/20, Loss: 1.0251923926475337
Epoch: 15/20, Loss: 1.0202786877016317
Epoch: 16/20, Loss: 1.0158154161254387
Epoch: 17/20, Loss: 1.0118822657206668
Epoch: 18/20, Loss: 1.0091486808430454
Epoch: 19/20, Loss: 1.0034196130897968
Epoch: 20/20, Loss: 1.001079035446783


In [26]:
torch.save(resnet34, 'resnet_emotion.pth')